In [18]:
import pandas as pd
import numpy as np
import warnings
from prettytable import PrettyTable
warnings.filterwarnings("ignore")

pd.set_option('display.max_colwidth', None)  # Display full content of each column
pd.set_option('display.max_columns', None)   # Display all columns
pd.set_option('display.width', 5000)         # Set display width

In [19]:
df = pd.read_excel("SpendWise2k26.xlsx")


In [20]:
na_count_per_column = df.applymap(lambda x: x == "N/A").sum()

# Print the result
print("Number of 'N/A' values per column:")
print(na_count_per_column)

Number of 'N/A' values per column:
Transaction_Date    0
Debit               0
Credit              0
Balance             0
Transaction_Mode    0
DR/CR_Indicator     0
Transaction_ID      0
Recipient_Name      0
Bank                0
UPI_ID              0
Note                0
Amount              0
dtype: int64


In [21]:
print(df.shape)
print(df.isna().sum())
# df.describe(include='object').T

(1653, 12)
Transaction_Date      0
Debit                 0
Credit                0
Balance               0
Transaction_Mode     36
DR/CR_Indicator       0
Transaction_ID        0
Recipient_Name       48
Bank                180
UPI_ID              180
Note                933
Amount                0
dtype: int64


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1653 entries, 0 to 1652
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_Date  1653 non-null   datetime64[ns]
 1   Debit             1653 non-null   float64       
 2   Credit            1653 non-null   float64       
 3   Balance           1653 non-null   float64       
 4   Transaction_Mode  1617 non-null   object        
 5   DR/CR_Indicator   1653 non-null   object        
 6   Transaction_ID    1653 non-null   object        
 7   Recipient_Name    1605 non-null   object        
 8   Bank              1473 non-null   object        
 9   UPI_ID            1473 non-null   object        
 10  Note              720 non-null    object        
 11  Amount            1653 non-null   float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 155.1+ KB


In [23]:
def categorize_transaction_frequency(filtered_transactions: pd.DataFrame):
    transaction_counts = filtered_transactions['Transaction_Key'].value_counts()
    
    frequency_groups = {
        '1 Transaction': 0, '1-5 Transactions': 0, '5-10 Transactions': 0, 
        '10-35 Transactions': 0, '35-50 Transactions': 0, 'More than 50': 0
    }
    
    for count in transaction_counts:
        if count == 1:
            frequency_groups['1 Transaction'] += 1
        elif 1 < count <= 5:
            frequency_groups['1-5 Transactions'] += 1
        elif 5 < count <= 10:
            frequency_groups['5-10 Transactions'] += 1
        elif 10 < count <= 35:
            frequency_groups['10-35 Transactions'] += 1
        elif 35 < count <= 50:
            frequency_groups['35-50 Transactions'] += 1
        else:
            frequency_groups['More than 50'] += 1
    
    table = PrettyTable()
    table.field_names = ['Category', 'Count']
    for category, count in frequency_groups.items():
        table.add_row([category, count])
    
    return table

def top_10_transactions(filtered_transactions: pd.DataFrame):
    transaction_counts = filtered_transactions['Transaction_Key'].value_counts().head(10)
    
    table = PrettyTable()
    table.field_names = ['Rank', 'Transaction_Key', 'Count']
    
    for i, (key, count) in enumerate(transaction_counts.items(), start=1):
        table.add_row([i, key, count])
    
    return table


In [24]:
def monthly_breakdown(transactions: pd.DataFrame):
    transactions['Transaction_Date'] = pd.to_datetime(transactions['Transaction_Date'])
    transactions['Month'] = transactions['Transaction_Date'].dt.to_period('M')
    monthly_summary = transactions.groupby('Month').agg({'Debit': 'sum', 'Credit': 'sum'}).reset_index()
    
    table = PrettyTable()
    table.field_names = ['Month', 'Total Spent (Debit)', 'Total Income (Credit)']
    for _, row in monthly_summary.iterrows():
        table.add_row([row['Month'], row['Debit'], row['Credit']])
    
    return table

def weekly_breakdown(transactions: pd.DataFrame):
    transactions['Week'] = transactions['Transaction_Date'].dt.to_period('W')
    weekly_summary = transactions.groupby('Week').agg({'Debit': 'sum', 'Credit': 'sum'}).reset_index()
    
    table = PrettyTable()
    table.field_names = ['Week', 'Total Spent (Debit)', 'Total Income (Credit)']
    for _, row in weekly_summary.iterrows():
        table.add_row([row['Week'], row['Debit'], row['Credit']])
    
    return table

def daily_breakdown(transactions: pd.DataFrame):
    daily_summary = transactions.groupby('Transaction_Date').agg({'Debit': 'sum', 'Credit': 'sum'}).reset_index()
    
    table = PrettyTable()
    table.field_names = ['Date', 'Total Spent (Debit)', 'Total Income (Credit)']
    for _, row in daily_summary.iterrows():
        table.add_row([row['Transaction_Date'], row['Debit'], row['Credit']])
    
    return table


In [25]:
def transaction_amount_ranges(filtered_transactions: pd.DataFrame):
    transaction_ranges = {
        '0-50': 0, '50-100': 0, '100-500': 0, '500-1000': 0, 
        '1000-5000': 0, '5000-10000': 0, '10000+': 0
    }
    
    for amount in filtered_transactions['Debit'].dropna():
        if amount <= 50:
            transaction_ranges['0-50'] += 1
        elif 50 < amount <= 100:
            transaction_ranges['50-100'] += 1
        elif 100 < amount <= 500:
            transaction_ranges['100-500'] += 1
        elif 500 < amount <= 1000:
            transaction_ranges['500-1000'] += 1
        elif 1000 < amount <= 5000:
            transaction_ranges['1000-5000'] += 1
        elif 5000 < amount <= 10000:
            transaction_ranges['5000-10000'] += 1
        else:
            transaction_ranges['10000+'] += 1
    
    table = PrettyTable()
    table.field_names = ['Transaction Range', 'Count']
    for category, count in transaction_ranges.items():
        table.add_row([category, count])
    
    return table

In [26]:
transactions=df
start_date='2022-04-01' 
end_date='2026-12-30'

filtered_transactions = transactions[(transactions['Transaction_Date'] >= start_date) &
                                     (transactions['Transaction_Date'] <= end_date)]
filtered_transactions['Transaction_Key'] = (
    filtered_transactions['Recipient_Name'].astype('object')
    .fillna(filtered_transactions['Transaction_ID'].astype('object'))
)
print("\nTransaction Frequency Distribution:")
print(categorize_transaction_frequency(filtered_transactions))

print("\nTop 10 Frequent Transactions:")
print(top_10_transactions(filtered_transactions))

print("\nTransaction Amount Distribution:")
print(transaction_amount_ranges(filtered_transactions))

print("\nMonthly Breakdown:")
print(monthly_breakdown(filtered_transactions))

# print("\nWeekly Breakdown:")
# print(weekly_breakdown(filtered_transactions))
    
# print("\nDaily Breakdown:")
# print(daily_breakdown(filtered_transactions))


Transaction Frequency Distribution:
+--------------------+-------+
|      Category      | Count |
+--------------------+-------+
|   1 Transaction    |  336  |
|  1-5 Transactions  |  117  |
| 5-10 Transactions  |   26  |
| 10-35 Transactions |   24  |
| 35-50 Transactions |   3   |
|    More than 50    |   3   |
+--------------------+-------+

Top 10 Frequent Transactions:
+------+-----------------+-------+
| Rank | Transaction_Key | Count |
+------+-----------------+-------+
|  1   |     Compass     |   86  |
|  2   |     JULFIKAR    |   62  |
|  3   |     PRACHI S    |   59  |
|  4   |     SNOW CRE    |   50  |
|  5   |    9890160567   |   48  |
|  6   |     Mr Vihaa    |   36  |
|  7   |     SUNIL SH    |   34  |
|  8   |     SHIVSAGA    |   29  |
|  9   |     Indian R    |   29  |
|  10  |     BIKANER     |   25  |
+------+-----------------+-------+

Transaction Amount Distribution:
+-------------------+-------+
| Transaction Range | Count |
+-------------------+-------+
|       

In [28]:
def print_transaction_details(df):
    """
    Prints every row with all the extracted details from the DataFrame.
    """
    for index, row in df.iterrows():
        print(f"Row {index}:")
        for col in df.columns:
            print(f"  {col}: {row[col]}")
print_transaction_details(df[:5])


Row 0:
  Transaction_Date: 2023-04-01 00:00:00
  Debit: 0.0
  Credit: 1500.0
  Balance: 1614.48
  Transaction_Mode: INB
  DR/CR_Indicator: CR
  Transaction_ID: IMPS309111290781
  Recipient_Name: 9890160567
  Bank: nan
  UPI_ID: nan
  Note: Son
  Amount: 1500.0
Row 1:
  Transaction_Date: 2023-04-01 00:00:00
  Debit: 75.0
  Credit: 0.0
  Balance: 1539.48
  Transaction_Mode: UPI
  DR/CR_Indicator: DR
  Transaction_ID: 309122218462
  Recipient_Name: ASIM HEM
  Bank: HDFC
  UPI_ID: asimshah
  Note: UPI
  Amount: -75.0
Row 2:
  Transaction_Date: 2023-04-02 00:00:00
  Debit: 1102.0
  Credit: 0.0
  Balance: 437.48
  Transaction_Mode: UPI
  DR/CR_Indicator: DR
  Transaction_ID: 309252494278
  Recipient_Name: SAI RAJ 
  Bank: PYTM
  UPI_ID: paytmqr2
  Note: UPI
  Amount: -1102.0
Row 3:
  Transaction_Date: 2023-04-02 00:00:00
  Debit: 10.0
  Credit: 0.0
  Balance: 427.48
  Transaction_Mode: UPI
  DR/CR_Indicator: DR
  Transaction_ID: 309255302589
  Recipient_Name: PRACHI S
  Bank: SBIN
  UPI_ID: 